In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql.functions import *
from delta.tables import DeltaTable

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
version_path = "output/delta/version_demo"

version_data = [
    (101, "Laptop", 55000),
    (102, "Mobile", 25000),
    (103, "Chair", 3000)
]

version_df = spark.createDataFrame(
    version_data,
    ["OrderID", "Product", "Price"]
)

version_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(version_path)

In [ ]:
# Create Version 1

version_table = DeltaTable.forPath(
    spark,
    version_path
)

version_table.update(
    condition=col("OrderID") == 101,
    set={"Price": lit(52000)}
)

In [ ]:
# Create Version 2

version_table.update(
    condition=col("OrderID") == 102,
    set={"Price": lit(24000)}
)

In [ ]:
# Read Version 0

spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(version_path).show()

# Read Version 1

spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load(version_path).show()

# Read Version 2

spark.read.format("delta") \
    .option("versionAsOf", 2) \
    .load(version_path).show()

In [ ]:
old_timestamp = version_table.history() \
    .filter(col("version") == 0) \
    .select("timestamp") \
    .first()["timestamp"]

In [ ]:
historical_df = spark.read \
    .format("delta") \
    .option("timestampAsOf", old_timestamp) \
    .load(version_path)

historical_df.show()

In [ ]:
version_table.restoreToVersion(0)